import warnings

warnings.filterwarnings(
    "ignore",
    message=r"Pandas requires version.*of 'numexpr'.*",
    category=UserWarning,
)# Predicting experimental Curie-Temperatures from compound embeddings 

This pipeline trains machine learning models that predict experimental Curie temperatures (Tc_exp, in Kelvin) directly from stoichiometric compound embeddings — without any simulated Tc values or data augmentation.

In [1]:
import sys
from pathlib import Path
import os

# Get project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent.resolve()

# Add project root to Python path so src/ is importable
sys.path.insert(0, str(PROJECT_ROOT))

# Change working directory to project root
os.chdir(PROJECT_ROOT)

### 1. Pre-Process Data

The preprocessing pipeline in this script prepares experimental and simulated Curie temperature (Tc) data.

1. **Aggregate** data from multiple sources.  
2. **Clean** Tc values: remove units, symbols, and uncertainties; convert to float.  
3. **Drop** invalid (non-numeric) Tc entries.  
4. **Deduplicate** by taking the median Tc per composition.  
5. **Flag** compositions containing rare-earth elements.  
6. **Split** data into RE-containing and RE-free subsets.  
7. **Save** clean, structured datasets for analysis.

In [2]:
from src.process_tc_data import main

In [3]:
main()

Experimental  — total: 14480, duplicates: 0
Simulated     — total: 2107, duplicates: 0


### 2. Embedding Creation

1. **Load Element Embeddings**:  
   Read pre-trained element vectors (e.g., Matscholar200) from a JSON file.

2. **Generate Compound Embeddings**:  
   For each composition, compute a weighted average of its constituent element embeddings, where weights are based on atomic fractions.

3. **Handle Missing Elements**:  
   Skip compounds containing elements not in the embedding dictionary.

4. **Filter Invalid Embeddings**:  
   Remove compounds that couldn’t be embedded (e.g., due to unknown elements).

5. **Save Results**:  
   Store the resulting DataFrame (with embeddings) as a pickle file for each dataset.

In [4]:
from src.create_embeddings import create_embeddings

In [5]:
create_embeddings()

Creating compound embeddings for experimental Tc datasets

Element embeddings: /raven/ptmp/cwinkler/ML-models-git/compound-to-tc/compound-to-experimental-tc/data/embeddings/element/matscholar200.json
Vocabulary: 103 elements  |  dimension: 200

------------------------------------------------------------
Dataset : RE-Free  (Experimental_Tc_RE-Free.csv)
  Rows with valid Tc_exp : 5680
  Dropped (un-embeddable): 34
  Embeddable rows        : 5646
  Saved → /raven/ptmp/cwinkler/ML-models-git/compound-to-tc/compound-to-experimental-tc/outputs/Experimental_Tc_RE-Free_w_embeddings.pkl

------------------------------------------------------------
Dataset : RE  (Experimental_Tc_RE.csv)
  Rows with valid Tc_exp : 8800
  Dropped (un-embeddable): 19
  Embeddable rows        : 8781
  Saved → /raven/ptmp/cwinkler/ML-models-git/compound-to-tc/compound-to-experimental-tc/outputs/Experimental_Tc_RE_w_embeddings.pkl

------------------------------------------------------------
Dataset : All  (Experimen

### 3. Compress Embeddings with PCA

1. **Load Pre-Computed Embeddings**:  
   Reads compound embeddings from pickle files (for RE-free, RE-containing, and All experimental datasets).

2. **Apply PCA**:  
   Compresses high-dimensional embeddings (e.g., 200+ dims) into lower-dimensional representations using PCA with 8, 16, 32, and 64 components.

3. **Preserve Explained Variance**:  
   Tracks how much variance each reduced dimension captures (typically >90% with 64 components).

4. **Save Compressed Embeddings**:  
   Adds new columns (`comp_emb_pca_8`, `comp_emb_pca_16`, etc.) to the DataFrame and saves the result as a new pickle file.

In [6]:
from src.compress_embeddings_pca import compress_embeddings_pca

In [7]:
compress_embeddings_pca()

PCA compression of compound embeddings
Component sizes: [8, 16, 32, 64]

------------------------------------------------------------
Dataset : RE-Free
  Loaded 5646 rows
  Raw embeddings shape: (5646, 200)
  PCA( 8 components): explained variance = 0.748  → 'comp_emb_pca_8'
  PCA(16 components): explained variance = 0.884  → 'comp_emb_pca_16'
  PCA(32 components): explained variance = 0.975  → 'comp_emb_pca_32'
  PCA(64 components): explained variance = 1.000  → 'comp_emb_pca_64'
  Saved → /raven/ptmp/cwinkler/ML-models-git/compound-to-tc/compound-to-experimental-tc/outputs/Experimental_Tc_RE-Free_w_embeddings_PCA.pkl

------------------------------------------------------------
Dataset : RE
  Loaded 8781 rows
  Raw embeddings shape: (8781, 200)
  PCA( 8 components): explained variance = 0.819  → 'comp_emb_pca_8'
  PCA(16 components): explained variance = 0.916  → 'comp_emb_pca_16'
  PCA(32 components): explained variance = 0.978  → 'comp_emb_pca_32'
  PCA(64 components): explained va

### 4. Model Training

1. **Input**:  
   Preprocessed experimental datasets with PCA-compressed compound embeddings (8–64D) and Tc_exp targets.

2. **Models Trained per Dataset** (RE-Free, RE, All):  
   - **Linear** (LassoLars, Ridge)  
   - **Random Forest**  
   - **MLP (Neural Network)**  
   All trained on raw 200D and PCA-reduced embeddings (8, 16, 32, 64D).

3. **Hyperparameter Tuning**:  
   - Randomized search (RF, MLP) and grid search (linear).  
   - Search space scaled to dataset size (e.g., fewer iterations for larger datasets).

4. **Evaluation & Visualization**:  
   - Metrics: R², MAE, RMSE.  
   - Plots: Predicted vs. actual Tc for train/test sets.

5. **Model Export**:  
   - Best models saved as ONNX (with preprocessing included).

6. **Output**:  
   - Per-dataset results (`results/<dataset>_results.csv`).  
   - Global comparison and best model summary (`exp_tc_comparison.csv`, `exp_tc_best_by_dataset.csv`).  
   - Figures and ONNX models for analysis and inference.

In [8]:
from src.train_exp_tc_all import main as main_all

[train_exp_tc] using n_jobs=8 for joblib/loky parallelism


In [9]:
# main_all()       # All (combined) dataset

In [10]:
from src.train_exp_tc_re import main as main_re

In [11]:
main_re()

Training (RE): compound embedding → experimental Tc

Dataset : RE  (Experimental_Tc_RE_w_embeddings_PCA.pkl)
Loaded 8781 rows
  RE physics features: ON (7 cols, 8090/8781 rows with RE content)

  [raw_200D]  X: (8781, 207)  train: ~7024  test: ~1757
    Training Linear (ensemble 1/10)...    Figure: RE_raw_200D_e0_linear.png
R²=0.5538  RMSE=182.5 K  MAE=138.7 K
  ONNX → RE_raw_200D_linear_refeats_e0.onnx
    Training Linear (ensemble 2/10)...    Figure: RE_raw_200D_e1_linear.png
R²=0.5515  RMSE=178.2 K  MAE=137.2 K
  ONNX → RE_raw_200D_linear_refeats_e1.onnx
    Training Linear (ensemble 3/10)...    Figure: RE_raw_200D_e2_linear.png
R²=0.5909  RMSE=169.8 K  MAE=130.9 K
  ONNX → RE_raw_200D_linear_refeats_e2.onnx
    Training Linear (ensemble 4/10)...    Figure: RE_raw_200D_e3_linear.png
R²=0.5968  RMSE=173.1 K  MAE=132.4 K
  ONNX → RE_raw_200D_linear_refeats_e3.onnx
    Training Linear (ensemble 5/10)...    Figure: RE_raw_200D_e4_linear.png
R²=0.5732  RMSE=178.3 K  MAE=134.5 K
  ONNX → 

Exception ignored in: <function ResourceTracker.__del__ at 0x1476dc2c2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x146a89ac2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

  Figure: RE_raw_200D_e1_rf.png
R²=0.9142  RMSE=77.9 K  MAE=45.1 K
  ONNX → RE_raw_200D_rf_refeats_e1.onnx
    Training RF (ensemble 3/10)...    Figure: RE_raw_200D_e2_rf.png
R²=0.9255  RMSE=72.5 K  MAE=42.5 K
  ONNX → RE_raw_200D_rf_refeats_e2.onnx
    Training RF (ensemble 4/10)...    Figure: RE_raw_200D_e3_rf.png
R²=0.9089  RMSE=82.3 K  MAE=45.0 K
  ONNX → RE_raw_200D_rf_refeats_e3.onnx
    Training RF (ensemble 5/10)...    Figure: RE_raw_200D_e4_rf.png
R²=0.9228  RMSE=75.8 K  MAE=43.7 K
  ONNX → RE_raw_200D_rf_refeats_e4.onnx
    Training RF (ensemble 6/10)...    Figure: RE_raw_200D_e5_rf.png
R²=0.9220  RMSE=75.1 K  MAE=44.3 K


Exception ignored in: <function ResourceTracker.__del__ at 0x152c96ec2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  ONNX → RE_raw_200D_rf_refeats_e5.onnx
    Training RF (ensemble 7/10)...    Figure: RE_raw_200D_e6_rf.png
R²=0.9379  RMSE=68.8 K  MAE=41.4 K


...

  Figure: RE_raw_200D_e6_mlp.png
R²=0.9179  RMSE=79.1 K  MAE=51.7 K
  ONNX → RE_raw_200D_mlp_refeats_e6.onnx
    Training MLP (ensemble 8/10)...    Figure: RE_raw_200D_e7_mlp.png
R²=0.9170  RMSE=76.0 K  MAE=49.6 K
  ONNX → RE_raw_200D_mlp_refeats_e7.onnx
    Training MLP (ensemble 9/10)...    Figure: RE_raw_200D_e8_mlp.png
R²=0.9156  RMSE=78.2 K  MAE=50.4 K
  ONNX → RE_raw_200D_mlp_refeats_e8.onnx
    Training MLP (ensemble 10/10)...    Figure: RE_raw_200D_e9_mlp.png
R²=0.9240  RMSE=75.9 K  MAE=48.6 K
  ONNX → RE_raw_200D_mlp_refeats_e9.onnx
    LightGBM not installed – skipping (pip install lightgbm).

  [pca_8]  X: (8781, 15)  train: ~7024  test: ~1757
    Training Linear (ensemble 1/10)...    Figure: RE_pca_8_e0_linear.png
R²=0.5226  RMSE=188.8 K  MAE=144.9 K
    ONNX export skipped (RE features: only raw_200D is ONNX-exportable (PCA variants need a ColumnTransformer))
    Training Linear (ensemble 2/10)...    Figure: RE_pca_8_e1_linear.png
R²=0.5117  RMSE=185.9 K  MAE=144.1 K
    O

Exception ignored in: <function ResourceTracker.__del__ at 0x148b8c8c2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x15270ddc2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

  Figure: RE_pca_64_e8_mlp.png
R²=0.9140  RMSE=79.0 K  MAE=50.0 K
    ONNX export skipped (RE features: only raw_200D is ONNX-exportable (PCA variants need a ColumnTransformer))
    Training MLP (ensemble 10/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x15517beb2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  Figure: RE_pca_64_e9_mlp.png
R²=0.9034  RMSE=85.6 K  MAE=52.0 K
    ONNX export skipped (RE features: only raw_200D is ONNX-exportable (PCA variants need a ColumnTransformer))
    LightGBM not installed – skipping (pip install lightgbm).

──────────────────────────────────────────────────────────────────────
Results for dataset: RE  (ensemble mean ± std over N members)
──────────────────────────────────────────────────────────────────────
Dataset Embedding  Model  N       R2   R2_std        MAE  MAE_std       RMSE  RMSE_std
     RE  raw_200D     RF 10 0.924596 0.009988  43.420516 1.702203  74.086363  4.833301
     RE    pca_32     RF 10 0.919606 0.009293  45.414275 1.588522  76.532365  4.218232
     RE    pca_16     RF 10 0.919051 0.009759  45.227501 1.686322  76.783388  4.427772
     RE    pca_64     RF 10 0.914744 0.008840  47.777508 1.452109  78.833209  3.829797
     RE     pca_8     RF 10 0.913488 0.009105  46.796400 1.461307  79.416061  4.090955
     RE    pca_32    MLP 10 0.908

In [12]:
from src.train_exp_tc_re_free import main as main_re_free

In [ ]:
main_re_free()

Training (RE-Free): compound embedding → experimental Tc

Dataset : RE-Free  (Experimental_Tc_RE-Free_w_embeddings_PCA.pkl)
Loaded 5646 rows
  RE physics features: ON (7 cols, 0/5646 rows with RE content)

  [raw_200D]  X: (5646, 207)  train: ~4516  test: ~1130
    Training Linear (ensemble 1/10)...    Figure: RE-Free_raw_200D_e0_linear.png
R²=0.3845  RMSE=204.7 K  MAE=154.4 K
  ONNX → RE-Free_raw_200D_linear_refeats_e0.onnx
    Training Linear (ensemble 2/10)...    Figure: RE-Free_raw_200D_e1_linear.png
R²=0.3922  RMSE=206.9 K  MAE=159.9 K
  ONNX → RE-Free_raw_200D_linear_refeats_e1.onnx
    Training Linear (ensemble 3/10)...    Figure: RE-Free_raw_200D_e2_linear.png
R²=0.3781  RMSE=207.9 K  MAE=157.3 K
  ONNX → RE-Free_raw_200D_linear_refeats_e2.onnx
    Training Linear (ensemble 4/10)...    Figure: RE-Free_raw_200D_e3_linear.png
R²=0.3855  RMSE=203.9 K  MAE=155.3 K
  ONNX → RE-Free_raw_200D_linear_refeats_e3.onnx
    Training Linear (ensemble 5/10)...    Figure: RE-Free_raw_200D_e4_

Exception ignored in: <function ResourceTracker.__del__ at 0x145e114b2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1468958be700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

  Figure: RE-Free_raw_200D_e1_mlp.png
R²=0.6473  RMSE=157.6 K  MAE=105.0 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e1.onnx
    Training MLP (ensemble 3/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x151261ec2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x14d04d4b2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

  Figure: RE-Free_raw_200D_e2_mlp.png
R²=0.5744  RMSE=172.0 K  MAE=115.5 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e2.onnx
    Training MLP (ensemble 4/10)...  

...

  Figure: RE-Free_raw_200D_e6_mlp.png
R²=0.6379  RMSE=154.9 K  MAE=103.2 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e6.onnx
    Training MLP (ensemble 8/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x151e489be700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  Figure: RE-Free_raw_200D_e7_mlp.png
R²=0.6559  RMSE=154.4 K  MAE=104.9 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e7.onnx
    Training MLP (ensemble 9/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x145ee51b2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  Figure: RE-Free_raw_200D_e8_mlp.png
R²=0.5826  RMSE=164.3 K  MAE=113.7 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e8.onnx
    Training MLP (ensemble 10/10)...    Figure: RE-Free_raw_200D_e9_mlp.png
R²=0.6659  RMSE=153.1 K  MAE=107.5 K
  ONNX → RE-Free_raw_200D_mlp_refeats_e9.onnx
    LightGBM not installed – skipping (pip install lightgbm).

  [pca_8]  X: (5646, 15)  train: ~4516  test: ~1130
    Training Linear (ensemble 1/10)...    Figure: RE-Free_pca_8_e0_linear.png
R²=0.3181  RMSE=215.5 K  MAE=164.4 K
    ONNX export skipped (RE features: only raw_200D is ONNX-exportable (PCA variants need a ColumnTransformer))
    Training Linear (ensemble 2/10)...    Figure: RE-Free_pca_8_e1_linear.png
R²=0.3353  RMSE=216.3 K  MAE=169.4 K
    ONNX export skipped (RE features: only raw_200D is ONNX-exportable (PCA variants need a ColumnTransformer))
    Training Linear (ensemble 3/10)...    Figure: RE-Free_pca_8_e2_linear.png
R²=0.3335  RMSE=215.2 K  MAE=165.1 K
    ONNX export skipped (RE features